In [ ]:
import pandas as pd

df = pd.DataFrame({
    'ClientId': [1, 1, 2, 2, 3, 4, 4, 4],
    'EventId': [1, 2, 1, 2, 1, 1, 2, 3],
    'Items': [['D'], ['A','B'], ['A','B'], ['A'], ['A','D'], ['D'], ['A','B'], ['A']]
})

df

,ClientId,EventId,Items
0,1,1,[D]
1,1,2,"[A, B]"
2,2,1,"[A, B]"
3,2,2,[A]
4,3,1,"[A, D]"
5,4,1,[D]
6,4,2,"[A, B]"
7,4,3,[A]


In [ ]:
from itertools import combinations
from collections import defaultdict


def get_support_count(candidate, database):
    count = 0
    for seq in database:
        i = 0
        for itemset in seq:
            if set(candidate[i]).issubset(set(itemset)):
                i += 1
                if i == len(candidate):
                    count += 1
                    break
    return count


def generate_candidates(all_freq):
    candidates = []
    for s1 in all_freq:
        for s2 in all_freq:
            if s1[-1] == s2[0]:  # Check if last item of s1 matches first item of s2
                if len(s2)==1:
                    new_cand = s1 + [s2[0]]  # Join s1 and s2
                else:
                    new_cand = s1 + s2[1:]  # Join s1 and s2 (excluding the first item of s2)
                candidates.append(new_cand)
    return candidates


def gsp(database, min_support_count):
    all_sequences_dict = {}

    support_dict = defaultdict(int)
    for seq in database:
        seen = set()
        for itemset in seq:
            # single items
            for item in itemset:
                seen.add(((item,),))
            # combinations inside same event
            for k in range(2, len(itemset)+1):
                for combo in combinations(itemset, k):
                    seen.add((tuple(combo),))
        # update support dict
        for pattern in seen:
            support_dict[pattern] += 1

    S1 = {}
    for pattern, count in support_dict.items():
        if count >= min_support_count:
            S1[pattern] = count
    all_sequences_dict.update(S1)
    # print("Frequent 1-itemsets:", S1)

    S2 = {}
    for s1 in S1.keys():
        for s2 in S1.keys():
            s1_list = list(s1)
            s2_list = list(s2)
            seq = s1_list + s2_list
            count = get_support_count(seq, database)
            if count >= min_support_count:
                S2[tuple(seq)] = count
    all_sequences_dict.update(S2)
    # print("Frequent 2-itemsets:", S2)

    prev_seq = S2
    while prev_seq:
        candidates = generate_candidates(prev_seq)
        new_seq = {}
        for cand in candidates:
            count = get_support_count(cand, database)
            if count >= min_support_count:
                new_seq[tuple(cand)] = count
        if not new_seq:
            break
        all_sequences_dict.update(new_seq)
        prev_seq = new_seq

    return all_sequences_dict

In [ ]:
database = df.groupby('ClientId')['Items'].apply(list).tolist()
database

[[['D'], ['A', 'B']],
 [['A', 'B'], ['A']],
 [['A', 'D']],
 [['D'], ['A', 'B'], ['A']]]

In [ ]:
min_confidence = 0.5
no_transactions = len(database)
min_support = 0.25
min_support_count = int(min_support * len(database))

all_sequences_dict = gsp(database, min_support_count)

# Print the sequences
all_sequences = list(all_sequences_dict.keys())
print("Frequent Sequences:")
for seq in all_sequences_dict:
    sequence = '->'.join(['{' + ','.join(itemset) + '}' for itemset in seq])
    print(sequence)

Frequent Sequences:
{A,B}
{B}
{D}
{A}
{A,D}
{A,B}->{A}
{B}->{A}
{D}->{A,B}
{D}->{B}
{D}->{A}
{A}->{A}
{D}->{A,B}->{A}
{D}->{B}->{A}
{D}->{A}->{A}


In [1]:
# Association rule mining

def support(itemset):
    return all_sequences_dict.get(itemset,0)/no_transactions

def get_confidence(antecedent, consequent):
    union = tuple(antecedent + consequent)
    antecedent_support = support(antecedent)
    if not antecedent_support:
        return 0
    return support(union) / antecedent_support

def get_association_rules(sequences, min_confidence):
    association_rules = []
    for seq,_ in sequences.items():
        for i in range(1, len(seq)):
            antecedent = seq[:i]
            consequent = seq[i:]
            conf = get_confidence(antecedent, consequent)
            if conf >= min_confidence:
                rule = f"{'->'.join(['{' + ','.join(itemset) + '}' for itemset in antecedent])} => {'->'.join(['{' + ','.join(itemset) + '}' for itemset in consequent])}"
                association_rules.append(rule)

    return association_rules


association_rules = get_association_rules(all_sequences_dict, min_confidence)
print("Association Rules:")
for rule in association_rules:
    print(rule)

Sequence Database:
S1: [['D'], ['A', 'B']]
S2: [['A', 'B'], ['A']]
S3: [['A', 'D']]
S4: [['D'], ['A', 'B'], ['A']]

Frequent Sequences:
{D}  (Support: 3)
{A}  (Support: 4)
{B}  (Support: 3)
{A,B}  (Support: 3)
{A,D}  (Support: 1)
{A}->{A}  (Support: 2)

Association Rules:
{A} => {A}  (Confidence: 0.5)
